# Tutorial 4: Using the Railway Pattern

The Railway Pattern enables conditional error chaining - automatically detecting follow-up errors based on conditions.

## What is the Railway Pattern?

When one error occurs, it often indicates that related errors may follow. The Railway Pattern allows you to:
- Chain error detection based on conditions
- Create branching logic for different scenarios
- Track cascading failures
- Build causal dependency graphs

## Condition Types

1. **always**: Always check next error
2. **field_equals**: Check if field equals value
3. **field_contains**: Check if field contains substring
4. **field_regex**: Match field against regex
5. **and**: All conditions must be true
6. **or**: Any condition must be true
7. **custom**: Custom Python function

## Example: Memory Exhaustion Chain

In [ ]:
import yaml
from datetime import datetime

railway_catalog = {
    "version": "1.0.0",
    "schema_version": "1.0.0",
    "metadata": {
        "name": "Memory Exhaustion Chain",
        "description": "Track memory issues from warning to failure",
        "author": "Tutorial",
        "created": datetime.now().isoformat(),
        "updated": datetime.now().isoformat()
    },
    "errors": {
        "high_memory_warning": {
            "id": "high_memory_warning",
            "pattern": {
                "type": "regex",
                "pattern": r"Memory usage (above|at|over) \d+%"
            },
            "files": ["/var/log/**/*.log"],
            "meaning": "Memory usage approaching limits",
            "suggestion": "Monitor for OOM events",
            "context_lines": 3,
            "next_errors": [
                {
                    "error_id": "oom_killer",
                    "when": {"type": "always"}  # Always check for OOM after high memory
                }
            ],
            "metadata": {"severity": "warning"}
        },
        "oom_killer": {
            "id": "oom_killer",
            "pattern": {
                "type": "literal",
                "pattern": "oom-kill"
            },
            "files": ["/var/log/**/*.log"],
            "meaning": "Process killed by OOM killer",
            "suggestion": "Increase memory allocation",
            "context_lines": 5,
            "next_errors": [
                {
                    "error_id": "kernel_panic",
                    "when": {
                        "type": "field_equals",
                        "field": "metadata.severity",
                        "value": "critical"
                    }
                }
            ],
            "metadata": {"severity": "critical"}
        },
        "kernel_panic": {
            "id": "kernel_panic",
            "pattern": {
                "type": "literal",
                "pattern": "Kernel panic"
            },
            "files": ["/var/log/**/*.log"],
            "meaning": "System crashed",
            "suggestion": "Review system logs and hardware",
            "context_lines": 10,
            "next_errors": [],
            "metadata": {"severity": "critical"}
        }
    }
}

print("Railway Pattern Chain: high_memory_warning → oom_killer → kernel_panic")
print("\nError chain configuration:")
print(yaml.dump(railway_catalog['errors']['high_memory_warning']['next_errors'], 
                default_flow_style=False, sort_keys=False))

## Conditional Branching Example

Different actions based on error context:

In [ ]:
branching_example = {
    "database_error": {
        "id": "database_error",
        "pattern": {
            "type": "regex",
            "pattern": r"Database error|SQLException"
        },
        "files": ["/var/log/app/**/*.log"],
        "meaning": "Database operation failed",
        "suggestion": "Check database connection and logs",
        "context_lines": 3,
        "next_errors": [
            {
                "error_id": "connection_timeout",
                "when": {
                    "type": "field_contains",
                    "field": "matched_text",
                    "value": "timeout"
                }
            },
            {
                "error_id": "deadlock",
                "when": {
                    "type": "field_contains",
                    "field": "matched_text",
                    "value": "deadlock"
                }
            },
            {
                "error_id": "connection_refused",
                "when": {
                    "type": "field_contains",
                    "field": "matched_text",
                    "value": "refused"
                }
            }
        ],
        "metadata": {"severity": "high", "category": "database"}
    }
}

print("Branching Logic:")
print("  database_error →")
print("    ├─ connection_timeout (if 'timeout' in matched_text)")
print("    ├─ deadlock (if 'deadlock' in matched_text)")
print("    └─ connection_refused (if 'refused' in matched_text)")

## Complex Conditions with AND/OR

Combine multiple conditions:

In [ ]:
complex_condition = {
    "application_error": {
        "id": "application_error",
        "pattern": {"type": "regex", "pattern": "ERROR"},
        "files": ["/var/log/**/*.log"],
        "meaning": "Application error",
        "suggestion": "Review error context",
        "context_lines": 3,
        "next_errors": [
            {
                "error_id": "escalate_to_oncall",
                "when": {
                    "type": "and",
                    "conditions": [
                        {
                            "type": "field_equals",
                            "field": "metadata.severity",
                            "value": "critical"
                        },
                        {
                            "type": "or",
                            "conditions": [
                                {
                                    "type": "field_contains",
                                    "field": "matched_text",
                                    "value": "production"
                                },
                                {
                                    "type": "field_contains",
                                    "field": "matched_text",
                                    "value": "customer-facing"
                                }
                            ]
                        }
                    ]
                }
            }
        ],
        "metadata": {"severity": "critical"}
    }
}

print("Complex Condition:")
print("  Escalate if (severity == 'critical') AND (environment is production OR customer-facing)")
print("\nCondition structure:")
print(yaml.dump(complex_condition['application_error']['next_errors'][0]['when'],
                default_flow_style=False, sort_keys=False))

## Best Practices

1. **Keep Chains Shallow**: Limit to 2-3 levels deep
2. **Avoid Cycles**: No circular dependencies (A → B → A)
3. **Document Conditions**: Use meaningful metadata
4. **Test Thoroughly**: Validate condition logic before deployment
5. **Use Specific Conditions**: Avoid overly broad matches

## Key Takeaways

- Railway Pattern enables conditional error chaining
- Multiple condition types for flexible logic
- Support for branching and complex boolean logic
- Essential for tracking cascading failures

## Next Steps

- [Tutorial 5: SSH Connection Pooling](05_ssh_pooling.ipynb)
- [Railway Pattern Deep Dive](../explanation/railway-pattern.md)
- [Condition Types Reference](../reference/condition-types.md)